# 서울시 119 구급 관련 공공 API 연동

전처리 노트북(`119_데이터_전처리.ipynb`)의 정적 파일 기반 데이터를 보완하기 위한
**실시간/최신 공공데이터 API 호출 코드**입니다.

| 절 | API | 상태 |
|---|---|---|
| 1 | 소방청 구급통계 서비스 (`EmergencyStatisticsService`) | 공식 가이드(docx)로 스펙 확인 완료 |
| 2 | 국립중앙의료원 응급의료기관 정보 조회 서비스 (`ErmctInfoInqireService`) | 가이드(hwp)가 DRM으로 잠혀 있어, 공공데이터포털 공식 페이지 기준으로 URL/파라미터만 검증 (응답 필드명은 코드가 전부 수집하도록 처리) |

필요 패키지: `pip install requests python-dotenv pandas`


## 0. 환경 설정

In [ ]:
import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = os.getenv("DATA_GO_KR_KEY")  # 공공데이터포털에서 발급받은 서비스키(디코딩 키)

# 원본 데이터 폴더 (자치구 매핑에 필요한 참고 파일 위치)
SRC_DIR = "C:/Users/Admin/Desktop/1st_project/data"
OUT_DIR = "C:/Users/Admin/Desktop/1st_project/data/outputs"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
class RateLimitExhausted(Exception):
    """
    당일 트래픽(일일 쿼터) 자체를 소진했을 가능성이 커서, 
    더 재시도해봐야 소용없다는 신호로 사용
    이걸 받으면 호출부는 즉시 전체 작업을 중단
    (추가 429로 시간/쿼터를 더 낭비하지 않기 위해).
    """
    pass


def request_get_with_retry(url, params, max_retries=5, base_delay=2.0, timeout=10):
    """
    429(요청 제한)/5xx 오류 시  재시도

    - 429 : Retry-After 헤더가 있으면 그 시간을, 없으면 base_delay * 2^시도횟수 만큼 대기 후 재시도
    - 5xx : 서버 일시 오류로 보고 동일하게 재시도
    - 그 외 4xx : 재시도해도 소용없으므로 즉시 예외 발생

    - max_retries를 다 썼는데도 429라면 : tps 제한이 아니라 당일 트래픽(쿼터) 소진일 가능성이 크므로
        RateLimitExhausted를 발생시켜, 호출부가 이 신호로 전체 배치 작업을 멈출 수 있게 한다.
    """
    last_status = None
    for attempt in range(max_retries):
        resp = requests.get(url, params=params, timeout=timeout)
        if resp.status_code == 200:
            return resp
        last_status = resp.status_code
        if resp.status_code == 429 or 500 <= resp.status_code < 600:
            wait = float(resp.headers.get("Retry-After", base_delay * (2 ** attempt)))
            print(f"  [재시도 {attempt+1}/{max_retries}] HTTP {resp.status_code} -> {wait:.1f}초 대기")
            time.sleep(wait)
            continue
        resp.raise_for_status()  # 429/5xx가 아닌 다른 오류는 바로 예외

    if last_status == 429:
        raise RateLimitExhausted(
            f"{max_retries}번 재시도했지만 계속 429입니다. tps 제한이 아니라 "
            "당일 트래픽(일일 쿼터)을 소진했을 가능성이 큽니다. "
            "공공데이터포털 마이페이지 > 활용현황에서 오늘 사용량을 확인해보세요."
        )
    resp.raise_for_status()  # 마지막 시도도 실패하면 예외 발생
    return resp


# 호출 간 기본 대기 시간(초). 공공데이터포털 대부분 API가 초당 트랜잭션(tps) 제한이 있어
# 너무 빠르게 반복 호출하면 429(Too Many Requests)가 발생한다. 429가 계속 뜨면 이 값을 늘릴 것.
REQUEST_INTERVAL = 1.0


## 소방서 → 자치구 매핑 준비

In [ ]:
import re

station_df = pd.read_csv(f"{SRC_DIR}/서울소방서 119안전센터 현황.csv", encoding="cp949")

def extract_gu(text):
    if pd.isna(text):
        return None
    m = re.search(r'([가-힣]+구)', str(text))
    return m.group(1) if m else None

station_df['District_guess'] = station_df['관할구역'].apply(extract_gu)
STATION2GU = station_df.groupby('서별')['District_guess'] \
    .agg(lambda x: x.value_counts().index[0]).to_dict()

VALID_GU = {'종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구','도봉구',
            '노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구','영등포구',
            '동작구','관악구','서초구','강남구','송파구','강동구'}
SEOUL_GU_LIST = sorted(VALID_GU)

print(f"소방서 매핑 {len(STATION2GU)}개, 자치구 {len(VALID_GU)}개")


---
## 1. 소방청 구급통계 서비스 (EmergencyStatisticsService)

In [ ]:
FIRE_BASE_URL = "http://apis.data.go.kr/1661000/EmergencyStatisticsService/get119EmgencyActivityStats"

def fetch_119_activity(sido_hq="서울소방재난본부", station=None, ym=None, num_of_rows=100, page_no=1):
    """소방청 구급통계 - 119구급활동현황 (get119EmgencyActivityStats) 호출"""
    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": page_no,
        "numOfRows": num_of_rows,
        "resultType": "xml",
        "sidoHqOgidNm": sido_hq,
    }
    if station:
        params["rsacGutFsttOgidNm"] = station
    if ym:
        params["rcptYm"] = ym  # 'YYYYMM' 형식

    resp = request_get_with_retry(FIRE_BASE_URL, params)

    root = ET.fromstring(resp.text)
    result_code = root.findtext(".//resultCode")
    if result_code != "00":
        raise RuntimeError(f"API 오류: {result_code} / {root.findtext('.//resultMsg')}")

    rows = []
    for item in root.findall(".//item"):
        rows.append({child.tag: child.text for child in item})
    return pd.DataFrame(rows), int(root.findtext(".//totalCount", default=0))


def fetch_119_activity_all(sido_hq="서울소방재난본부", station=None, ym=None):
    """totalCount 만큼 페이징하며 전체 수집"""
    all_rows = []
    page = 1
    while True:
        df, total = fetch_119_activity(sido_hq, station, ym, num_of_rows=100, page_no=page)
        if df.empty:
            break
        all_rows.append(df)
        if page * 100 >= total:
            break
        page += 1
        time.sleep(REQUEST_INTERVAL)
    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


# 사용 예시
df_202506 = fetch_119_activity_all(sido_hq="서울소방재난본부", ym="202506")
df_202506.head()


### 25개 자치구(소방서) 전체 최신 월 수집

In [ ]:
def collect_latest_month_all_gu(ym):
    """지정한 접수년월(YYYYMM)에 대해 25개 소방서 전체 데이터를 수집하여
    ambulance_prophet.csv와 동일한 스키마(ds, District, y, transport_cnt, transport_person)로 반환"""
    records = []
    for station, gu in STATION2GU.items():
        try:
            df = fetch_119_activity_all(sido_hq="서울소방재난본부", station=station, ym=ym)
        except Exception as e:
            print(f"[경고] {station} 수집 실패: {e}")
            time.sleep(REQUEST_INTERVAL)
            continue
        if df.empty:
            time.sleep(REQUEST_INTERVAL)
            continue
        row = df.iloc[0]
        records.append({
            "ds": f"{ym[:4]}-{ym[4:6]}-01",
            "District": gu,
            "y": float(row.get("gutCo", 0)),
            "transport_cnt": float(row.get("trnfCo", 0)),
            "transport_person": float(row.get("trnfPcnt", 0)),
        })
        time.sleep(REQUEST_INTERVAL)
    return pd.DataFrame(records)

# 사용 예시
latest = collect_latest_month_all_gu("202508")
latest.to_csv(f"{OUT_DIR}/ambulance_prophet_202508_api.csv", index=False, encoding='utf-8-sig')
latest.head()


In [ ]:
def collect_month_range_all_gu(start_ym, end_ym, combined_path=None, resume=True):
    """
    start_ym ~ end_ym(둘 다 'YYYYMM', 포함) 구간의 월별 구급활동을 전부 수집해
    ambulance_prophet.csv와 동일한 컬럼으로 하나의 DataFrame에 쌓아 반환

    - combined_path : 결과를 저장/이어받을 CSV 경로. 기본값은 OUT_DIR/ambulance_prophet_{start}_{end}_api.csv
    - resume=True면, combined_path에 이미 저장된 월(ds)은 다시 호출하지 않고 건너뛴다.
        (중간에 끊기거나 429로 실패해도 다시 실행하면 안 받은 달부터 이어서 받는다)
    - 일일 쿼터 소진으로 보이는 RateLimitExhausted를 만나면, 지금까지 모은 달까지는 이미
        매달 저장해뒀으므로 데이터 손실 없이 그 자리에서 멈춘다. 쿼터가 리셋된 뒤(보통 다음날)
        같은 셀을 다시 실행하면 안 받은 달부터 이어서 받는다.
    """
    if combined_path is None:
        combined_path = f"{OUT_DIR}/ambulance_prophet_{start_ym}_{end_ym}_api.csv"

    months = pd.period_range(
        start=pd.Period(start_ym, freq="M"),
        end=pd.Period(end_ym, freq="M"), 
        freq="M"
        )
    months = [m.strftime("%Y%m") for m in months]

    already = pd.DataFrame()
    done_months = set()
    if resume and os.path.exists(combined_path):
        already = pd.read_csv(combined_path)
        done_months = set(pd.to_datetime(already["ds"]).dt.strftime("%Y%m"))
        print(f"[이어받기] {combined_path}에 이미 {len(done_months)}개월치가 있어 건너뜁니다: {sorted(done_months)}")

    all_frames = [already] if not already.empty else []
    quota_exhausted = False
    for ym in months:
        if ym in done_months:
            continue
        print(f"=== {ym} 수집 중 ({months.index(ym)+1}/{len(months)}) ===")
        try:
            month_df = collect_latest_month_all_gu(ym)
        except RateLimitExhausted as e:
            print(f"\n[중단] 일일 쿼터 소진으로 추정되어 백필을 멈춥니다: {e}")
            print(f"지금까지 모은 달까지는 이미 {combined_path}에 저장되어 있습니다.")
            print("쿼터가 리셋된 뒤(보통 자정, 공공데이터포털 활용현황에서 확인) 같은 셀을 다시 실행하면 이어받습니다.")
            quota_exhausted = True
            break
        if month_df.empty:
            print(f"  -> {ym}: 데이터 없음(집계 전으로 추정), 건너뜀")
            continue
        all_frames.append(month_df)

        # 중간 저장: 매달 끝날 때마다 지금까지 결과를 저장해서, 도중에 실패해도 이어받을 수 있게 함
        combined_so_far = pd.concat(all_frames, ignore_index=True).drop_duplicates(subset=["ds", "District"])
        combined_so_far.to_csv(combined_path, index=False, encoding="utf-8-sig")
        print(f"  -> {ym}: {len(month_df)}행 수집, 누적 저장 완료 ({combined_path})")

    if not all_frames:
        if not quota_exhausted:
            print("[경고] 수집된 데이터가 전혀 없습니다.")
        return pd.DataFrame()

    result = pd.concat(all_frames, ignore_index=True).drop_duplicates(subset=["ds", "District"]) \
                .sort_values(["District", "ds"]).reset_index(drop=True)
    return result


# 사용 예시:
combined = collect_month_range_all_gu("202501", "202512")
combined.head()


In [ ]:
def _probe_month_has_data(ym, probe_station="구로소방서"):
    """단일 소방서 1건만 조회해서 해당 월에 데이터가 있는지 빠르게 확인"""
    try:
        probe_df, _ = fetch_119_activity(
            sido_hq="서울소방재난본부", 
            station=probe_station,
            ym=ym, 
            num_of_rows=1, 
            page_no=1)
    except RateLimitExhausted:
        raise
    except Exception as e:
        print(f"  {ym} probe 실패: {e}")
        return False
    return not probe_df.empty


def find_latest_available_month(max_back=36, probe_station="구로소방서"):
    """오늘 기준 전월부터 거슬러 올라가며 데이터가 있는 가장 최근 월을 탐색"""
    start_ym = pd.Timestamp.now().replace(day=1) - pd.Timedelta(days=1)
    for back in range(max_back):
        try_ym = (start_ym.replace(day=1) - pd.DateOffset(months=back)).strftime("%Y%m")
        if _probe_month_has_data(try_ym, probe_station):
            return try_ym
        time.sleep(REQUEST_INTERVAL)
    return None


def find_earliest_available_month(search_from="201001", max_forward=240, probe_station="구로소방서"):
    """search_from부터 앞으로 나아가며 데이터가 있는 가장 이른 월을 탐색"""
    ym = pd.Period(search_from, freq="M")
    for _ in range(max_forward):
        try_ym = ym.strftime("%Y%m")
        if _probe_month_has_data(try_ym, probe_station):
            return try_ym
        ym += 1
        time.sleep(REQUEST_INTERVAL)
    return None


---
## 2. 국립중앙의료원 응급의료기관 정보 조회 서비스

공공데이터포털 서비스ID `15000563`

| 필드(영문) | 의미 |
|---|---|
| `hpid` | 기관 ID (공통 조인 키) |
| `dutyName` | 기관명 |
| `dutyTel3` | 기관 대표전화 |
| `hvidate` | 병상정보 입력(갱신)일시 |
| `hvec` | **응급실 일반병상 가용 수** (= `Available_Beds`) |
| `hvamyn` | 구급차 수용 가능 여부 |


In [ ]:
NIA_BASE_URL = "http://apis.data.go.kr/B552657/ErmctInfoInqireService"

def _parse_items_generic(xml_text):
    """item의 모든 하위 태그를 그대로 dict로 변환 (원본 검증/디버깅용)"""
    root = ET.fromstring(xml_text)
    result_code = root.findtext(".//resultCode")
    if result_code not in (None, "00"):
        raise RuntimeError(f"API 오류: {result_code} / {root.findtext('.//resultMsg')}")
    rows = [{child.tag: child.text for child in item} for item in root.findall(".//item")]
    total = int(root.findtext(".//totalCount", default=0) or 0)
    return rows, total


def fetch_egyt_bass_info(sido, sigungu=None, num_of_rows=100, page_no=1):
    """응급의료기관 기본정보 조회 (getEgytBassInfoInqire) - 주소(dutyAddr) 등 확보용"""
    params = {"serviceKey": SERVICE_KEY, "pageNo": page_no, "numOfRows": num_of_rows, "Q0": sido}
    if sigungu:
        params["Q1"] = sigungu
    resp = request_get_with_retry(f"{NIA_BASE_URL}/getEgytBassInfoInqire", params)
    rows, total = _parse_items_generic(resp.text)
    return pd.DataFrame(rows), total


# 응급실(hvec) 관점에서 의미 있는 필드만 추출할 때 사용할 컬럼 매핑
ER_FIELD_MAP = {
    "hpid": "Hospital_ID",
    "dutyName": "Hospital_Name",
    "dutyTel3": "Tel",
    "hvidate": "Updated_At",
    "hvec": "Available_Beds",   # 응급실 일반병상 가용 수
    "hvamyn": "Ambulance_Available",
}

def fetch_realtime_er_beds(sido, sigungu=None, num_of_rows=100, page_no=1):
    """실시간 응급실 가용병상정보 조회에서
    '응급실' 관점 핵심 필드(ER_FIELD_MAP)만 뽑아서 반환

    이 API는 병상 수가 음수(예: -1)로 오면 '정보 없음/미보고' 상태를 의미하는
    특수 코드로 사용(실제로 병상이 마이너스라는 뜻이 아님 - 해당 병원이 그 시점에 병상
    현황을 보고하지 않았거나 시스템 점검 중일 때 나오는 값)
    
    이걸 그대로 두면 구 단위로
    합산할 때 실제 가용병상이 있는 다른 병원의 값까지 깎아먹어 전체가 마이너스로 보이는
    왜곡이 생기므로, 여기서 음수는 결측치(None)로 바꾸고 별도로 표시"""
    params = {
        "serviceKey": SERVICE_KEY,
        "STAGE1": sido,   
        "STAGE2": sigungu,   
        "pageNo": page_no,
        "numOfRows": num_of_rows,
    }
    resp = request_get_with_retry(f"{NIA_BASE_URL}/getEmrrmRltmUsefulSckbdInfoInqire", params)
    rows, total = _parse_items_generic(resp.text)

    cleaned = []
    for row in rows:
        item = {new_col: row.get(old_col) for old_col, new_col in ER_FIELD_MAP.items()}
        try:
            beds = int(item["Available_Beds"])
        except (TypeError, ValueError):
            beds = None

        if beds is not None and beds < 0:
            # 음수 = '정보 없음/미보고' 특수 코드. 실제 병상 수가 아니므로 결측 처리.
            item["Beds_Unknown"] = True
            item["Available_Beds"] = None
        else:
            item["Beds_Unknown"] = beds is None
            item["Available_Beds"] = beds
        cleaned.append(item)

    return pd.DataFrame(cleaned), total


# 사용 예시 (서비스키가 준비된 뒤 주석 해제):
er_df, total = fetch_realtime_er_beds(sido="서울특별시", sigungu="종로구")
er_df.head()


### 25개 자치구 전체 순회 + Forward Fill 처리

In [ ]:
def collect_realtime_beds_all_gu(prev_file=None):
    """
    25개 자치구를 순회하며 응급실 가용병상(Available_Beds)만 수집

    prev_file(기본값: OUT_DIR/realtime_er_beds.csv)이 존재하면, 이번 호출이 실패/빈 응답인 구는
    그 파일에 남아있는 해당 구의 마지막 값으로 ffill 처리

    컬럼: Hospital_ID, Hospital_Name, Tel, Updated_At, Available_Beds, Ambulance_Available,
        District, collected_at, is_ffilled
    """
    if prev_file is None:
        prev_file = f"{OUT_DIR}/realtime_er_beds.csv"

    prev_by_gu = {}
    if os.path.exists(prev_file):
        prev_df = pd.read_csv(prev_file)
        for gu, g in prev_df.groupby("District"):
            prev_by_gu[gu] = g

    frames = []
    for gu in SEOUL_GU_LIST:
        try:
            df, total = fetch_realtime_er_beds(sido="서울특별시", sigungu=gu)
            if df.empty:
                raise ValueError(f"빈 응답 (totalCount={total})")
            df["District"] = gu
            df["collected_at"] = pd.Timestamp.now()
            df["is_ffilled"] = False
            frames.append(df)
        except Exception as e:
            print(f"[경고] {gu} 실시간 병상 조회 실패: {e}")
            if gu in prev_by_gu:
                ffilled = prev_by_gu[gu].copy()
                ffilled["is_ffilled"] = True
                ffilled["collected_at"] = pd.Timestamp.now()
                frames.append(ffilled)
                print(f"  ㄴ {gu}: 이전 저장 파일({os.path.basename(prev_file)})의 값으로 대체")
            else:
                print(f"  ㄴ {gu}: 이전 저장 파일에도 데이터가 없어 건너뜀")
        time.sleep(REQUEST_INTERVAL)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# 사용 예시
realtime_beds = collect_realtime_beds_all_gu()
realtime_beds.to_csv(f"{OUT_DIR}/realtime_er_beds.csv", index=False, encoding='utf-8-sig')
realtime_beds.head()

---
## 3. 파일로 저장

In [ ]:
# 3-1. 소방청 구급통계 - 최신 월 25개 구 수집 및 저장

if not SERVICE_KEY:
    print("[중단] SERVICE_KEY가 없습니다. .env에 DATA_GO_KR_KEY를 설정한 뒤 셀 2(환경 설정)부터 다시 실행해주세요.")
else:
    try:
        target_ym = find_latest_available_month()
        if target_ym is None:
            print("[경고] 최근 구간에서 데이터를 찾지 못했습니다. find_latest_available_month(max_back=...)"
                    " 값을 늘려서 다시 시도해보세요.")
        else:
            print(f"{target_ym}: 데이터 확인됨 -> 25개 구 전체 수집 시작")
            latest = collect_latest_month_all_gu(target_ym)
            if latest.empty:
                print("[경고] probe에서는 데이터가 있었는데 전체 수집 결과가 비어있습니다. 잠시 후 다시 시도해보세요.")
            else:
                out_path = f"{OUT_DIR}/ambulance_prophet_{target_ym}_api.csv"
                latest.to_csv(out_path, index=False, encoding='utf-8-sig')
                print(f"저장 완료: {out_path}  ({len(latest)}행, 기준월={target_ym})")
                display(latest.head())
    except RateLimitExhausted as e:
        print(f"[중단] 일일 쿼터 소진으로 추정됩니다: {e}")
        print("공공데이터포털 마이페이지 > 활용현황에서 오늘 사용량을 확인하시고, 리셋된 뒤 다시 실행해주세요.")


### 3-1b. 있는 만큼 전부 수집 (예측 모델용 시계열 백필)

In [ ]:
# 3-1b. 실행: 있는 만큼 전부 자동 수집
EARLIEST_SEARCH_FROM = "201001"  # 이보다 더 과거까지 있을 것 같으면 이 값을 낮춰서 다시 탐색 가능

if not SERVICE_KEY:
    print("[중단] SERVICE_KEY가 없습니다. .env에 DATA_GO_KR_KEY를 설정한 뒤 셀 2(환경 설정)부터 다시 실행해주세요.")
else:
    try:
        print("데이터 보유 구간 탐색 중...")
        latest_ym = find_latest_available_month()
        print(f"가장 최근 데이터: {latest_ym}")
        earliest_ym = find_earliest_available_month(search_from=EARLIEST_SEARCH_FROM)
        print(f"가장 이른 데이터: {earliest_ym}")

        if latest_ym is None or earliest_ym is None:
            print("[경고] 데이터 보유 구간을 찾지 못했습니다. EARLIEST_SEARCH_FROM 값을 조정해서 다시 시도해주세요.")
        else:
            print(f"\n{earliest_ym} ~ {latest_ym} 전체 수집을 시작합니다 (시간이 꽤 걸릴 수 있습니다)...")
            combined = collect_month_range_all_gu(earliest_ym, latest_ym)
            if not combined.empty:
                print(f"\n최종 결과: {len(combined)}행, {combined['District'].nunique()}개 구, "
                    f"{combined['ds'].nunique()}개월 ({earliest_ym} ~ {latest_ym})")
                display(combined.head())
    except RateLimitExhausted as e:
        print(f"[중단] 경계 탐색 중 일일 쿼터 소진으로 추정됩니다: {e}")
        print("공공데이터포털 마이페이지 > 활용현황에서 오늘 사용량을 확인하시고, 리셋된 뒤 다시 실행해주세요.")


In [ ]:
# 3-2. 실시간 응급실 가용병상 - 25개 구 전체 수집 및 저장
if not SERVICE_KEY:
    print("[중단] SERVICE_KEY가 없습니다. .env에 DATA_GO_KR_KEY를 설정한 뒤 셀 2(환경 설정)부터 다시 실행해주세요.")
else:
    print("25개 구 실시간 응급실 가용병상 수집 중... (구별로 호출하므로 몇 초 정도 걸립니다)")
    realtime_beds = collect_realtime_beds_all_gu()
    if realtime_beds.empty:
        print("[경고] 수집된 데이터가 없습니다. 서비스키 활용신청이 '운영중' 상태인지 확인해주세요.")
    else:
        out_path = f"{OUT_DIR}/realtime_er_beds.csv"
        realtime_beds.to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f"저장 완료: {out_path}  ({len(realtime_beds)}행, ffill 적용 {realtime_beds['is_ffilled'].sum()}건)")
        display(realtime_beds.head())
